In [4]:
# Import and initialize poolparty
import poolparty as pp 
pp.init()

template_pool = pp.from_seq('TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGG</cre>ATTACGG<bc/>AGATCGGA')\
    .named('template_pool').stylize('cre', style='salmon')

mutated_pool = template_pool.mutagenize('cre',
                                        mutation_rate=0.1, 
                                        mode='random', 
                                        num_states=5, 
                                        mark_changes=False,
                                        style_mutations='yellow bold',
                                        seq_name_prefix='mut_').named('mutated_pool')

deletion_pool = template_pool.deletion_scan('cre', 
                                            deletion_length=5, 
                                            positions=slice(None, None, 5), 
                                            mode='sequential', 
                                            style_deletion='magenta',
                                            seq_name_prefix='del_').named('deletion_pool')

sites_pool=pp.from_seqs(['AAAAA','TTTTT'], 
                        mode='sequential', 
                        op_iter_order=-1,
                        seq_name_prefix='site_').named('sites_pool').stylize(style='cyan')

insertion_pool = template_pool.insertion_scan('cre', 
                                              ins_pool=sites_pool, 
                                              positions=slice(None,None,5), 
                                              replace=True, 
                                              mode='sequential',
                                              seq_name_pos_prefix='pos_', 
                                              seq_name_site_prefix='site_',
                                              seq_name_prefix='ins_').named('insertion_pool')

combo_pool = pp.stack([mutated_pool, deletion_pool, insertion_pool], name='stacked_pool')\
    .repeat_states(2, seq_name_prefix='v_', op_iter_order=-2, name='repeated_pool')\
    .insert_kmers('bc', mode='random', length=5, seq_name_prefix='bc_')\
    .named('combo_pool')\

combo_pool.print_library(show_name=True,seed=10)

combo_pool: seq_length=None, num_states=40
state  name                          seq
    0  mut_0.v_0.bc_0                TCCCGACT<cre>GGAAAGCTGGCAGTGAGTACACAGG</cre>ATTACGG<bc>gacac</bc>AGATCGGA
    1  mut_0.v_1.bc_1                TCCCGACT<cre>GGAAAGCTGGCAGTGAGTACACAGG</cre>ATTACGG<bc>aagcc</bc>AGATCGGA
    2  mut_1.v_0.bc_2                TCCCGACT<cre>TGAACGCGGGCAGTGAGCACACAGT</cre>ATTACGG<bc>cttcg</bc>AGATCGGA
    3  mut_1.v_1.bc_3                TCCCGACT<cre>TGAACGCGGGCAGTGAGCACACAGT</cre>ATTACGG<bc>attaa</bc>AGATCGGA
    4  mut_2.v_0.bc_4                TCCCGACT<cre>GGAAAGTGTGCAGTGAGTACACAGG</cre>ATTACGG<bc>accgg</bc>AGATCGGA
    5  mut_2.v_1.bc_5                TCCCGACT<cre>GGAAAGTGTGCAGTGAGTACACAGG</cre>ATTACGG<bc>aaaca</bc>AGATCGGA
    6  mut_3.v_0.bc_6                TCCCGACT<cre>GGAAAGCGGGCAGTGAGCCCACAGG</cre>ATTACGG<bc>aaaca</bc>AGATCGGA
    7  mut_3.v_1.bc_7                TCCCGACT<cre>GGAAAGCGGGCAGTGAGCCCACAGG</cre>ATTACGG<bc>gcttc</bc>AGATCGGA
    8  mut_4.v_0.bc_8       

Pool(id=13, name='combo_pool', op='op[13]:get_kmers', num_states=40)

In [2]:
combo_pool.state.print_dag()

combo_pool.state (o=-2, n=40)
├── op[12]:get_kmers.state (o=-2, n=40)
└── repeated_pool.state (o=-2, n=40)
    └── [op=Product]
        ├── op[11]:repeat.state (o=-2, n=2)
        └── stacked_pool.state (o=-1, n=20)
            └── [op=Stack]
                ├── mutated_pool.state (o=0, n=5)
                │   └── [op=Product]
                │       ├── op[2]:mutagenize.state (o=0, n=5)
                │       └── pool[1].state (o=0, n=1)
                │           └── template_pool.state (o=0, n=1)
                ├── deletion_pool.state (o=0, n=5)
                │   └── pool[3].state (o=0, n=5)
                │       └── [op=Product]
                │           ├── op[3]:deletion_scan(marker_scan).state (o=0, n=5)
                │           └── pool[1].state (o=0, n=1)
                └── insertion_pool.state (o=-1, n=10)
                    └── [op=Product]
                        ├── pool[7].state (o=-1, n=2)
                        │   ├── op[5]:from_seqs.state (o=-1, n=2)
 